In [1]:
import gymnasium as gym
import time

# 1. Crear el entorno de Acrobot
env = gym.make("Acrobot-v1", render_mode="human")



In [3]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

def discretize_state(state, bins):
    """
    Convierte estados continuos en índices discretos
    """
    state_indices = []

    for i in range(len(state)):
        state_indices.append(
            np.digitize(state[i], bins[i]) - 1
        )

    return tuple(state_indices)

def train(episodes):

    env = gym.make("Acrobot-v1")

    # Límites aproximados del entorno
    state_bounds = list(zip(env.observation_space.low,
                            env.observation_space.high))

    # Acrobot tiene valores infinitos en velocidades
    # Los reemplazamos manualmente
    state_bounds[4] = (-10, 10)
    state_bounds[5] = (-10, 10)

    # Número de divisiones por variable
    num_bins = 10

    bins = [
        np.linspace(state_bounds[i][0],
                    state_bounds[i][1],
                    num_bins)
        for i in range(len(state_bounds))
    ]

    # Tabla Q
    q_table = np.zeros((num_bins,
                        num_bins,
                        num_bins,
                        num_bins,
                        num_bins,
                        num_bins,
                        env.action_space.n))

    learning_rate = 0.1
    discount_factor = 0.99

    epsilon = 1.0
    epsilon_decay = 0.995
    epsilon_min = 0.01

    rng = np.random.default_rng()

    rewards_per_episode = np.zeros(episodes)

    for episode in range(episodes):

        # Render cada cierto tiempo
        if (episode + 1) % 500 == 0:
            env.close()
            env = gym.make("Acrobot-v1", render_mode="human")
        else:
            env.close()
            env = gym.make("Acrobot-v1")

        state = env.reset()[0]

        # Discretizar estado inicial
        state_disc = discretize_state(state, bins)

        terminated = False
        truncated = False

        total_reward = 0

        while not terminated and not truncated:

            # Exploración / explotación
            if rng.random() < epsilon:
                action = env.action_space.sample()
            else:
                action = np.argmax(q_table[state_disc])

            # Ejecutar acción
            new_state, reward, terminated, truncated, _ = env.step(action)

            # Discretizar nuevo estado
            new_state_disc = discretize_state(new_state, bins)

            # Q-Learning update
            q_table[state_disc][action] = (
                q_table[state_disc][action]
                + learning_rate * (
                    reward
                    + discount_factor * np.max(q_table[new_state_disc])
                    - q_table[state_disc][action]
                )
            )

            state_disc = new_state_disc

            total_reward += reward

        rewards_per_episode[episode] = total_reward

        # Decaimiento epsilon
        epsilon = max(epsilon * epsilon_decay, epsilon_min)

        # Mostrar progreso
        if (episode + 1) % 500 == 0:
            print(f"Episodio: {episode + 1}")
            print(f"Recompensa total: {total_reward}")
            print(f"Epsilon: {epsilon:.4f}")

    env.close()

    # Promedio móvil
    moving_avg = np.convolve(
        rewards_per_episode,
        np.ones(100)/100,
        mode='valid'
    )

    plt.plot(moving_avg)
    plt.title("Recompensa promedio")
    plt.xlabel("Episodios")
    plt.ylabel("Reward")
    plt.show()


if __name__ == "__main__":
    train(5000)

Episodio: 500
Recompensa total: -500.0
Epsilon: 0.0816
Episodio: 1000
Recompensa total: -500.0
Epsilon: 0.0100
Episodio: 1500
Recompensa total: -200.0
Epsilon: 0.0100
Episodio: 2000
Recompensa total: -406.0
Epsilon: 0.0100
Episodio: 2500
Recompensa total: -132.0
Epsilon: 0.0100


KeyboardInterrupt: 

In [4]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import time  # <--- Importamos time para controlar la velocidad visual

def train(episodes):
    # 1. Configuración de la discretización para Acrobot-v1
    BINS = [10, 10, 10, 10, 10, 10] 
    
    lower_bounds = [-1.0, -1.0, -1.0, -1.0, -12.5, -28.5]
    upper_bounds = [1.0, 1.0, 1.0, 1.0, 12.5, 28.5]
    
    state_bins = [np.linspace(lower_bounds[j], upper_bounds[j], BINS[j] - 1) for j in range(6)]

    def discretize_state(continuous_state):
        discretized = []
        for j in range(6):
            bin_index = int(np.digitize(continuous_state[j], state_bins[j]))
            discretized.append(bin_index)
        return tuple(discretized)

    # 2. Inicialización de la Tabla Q
    env = gym.make('Acrobot-v1')
    action_space_size = env.action_space.n
    q_table = np.zeros(BINS + [action_space_size])
             
    # Hiperparámetros
    learning_rate = 0.2
    discount_factor = 0.99
    epsilon = 1.0
    epsilon_decay_rate = 2 / episodes  
    rng = np.random.default_rng()
    
    rewards_per_episode = np.zeros(episodes)
    
    for i in range(episodes):
        # Bandera para saber si en este episodio vamos a renderizar visualmente
        rendering = False
        
        # Cada 500 episodios mostramos el entorno de forma visual
        if (i + 1) % 500 == 0:
            env.close()
            env = gym.make('Acrobot-v1', render_mode='human')
            rendering = True
        elif (i + 1) % 500 == 1 and i > 0:
            env.close()
            env = gym.make('Acrobot-v1')
            
        continuous_state, _ = env.reset()
        state = discretize_state(continuous_state)
        
        terminated = False
        truncated = False
        total_reward = 0
        
        # Bucle del episodio: se detiene al ganar (terminated) o agotar movimientos (truncated)
        while not terminated and not truncated:
            # Control de velocidad en la visualización
            if rendering:
                time.sleep(0.01) # Pausa de 10 milisegundos para poder apreciar los movimientos

            # Decisión de explorar o explotar
            if rng.random() < epsilon:
                action = env.action_space.sample()
            else:
                action = np.argmax(q_table[state])
                
            # Ejecutar acción
            new_continuous_state, reward, terminated, truncated, _ = env.step(action)
            new_state = discretize_state(new_continuous_state)
            
            total_reward += reward
            
            # Actualización de la Tabla Q
            best_future_q = np.max(q_table[new_state])
            q_table[state + (action,)] = q_table[state + (action,)] + learning_rate * (
                reward + discount_factor * best_future_q - q_table[state + (action,)]
            )
            
            state = new_state
            
        # Reducir epsilon
        epsilon = max(epsilon - epsilon_decay_rate, 0.01)
        rewards_per_episode[i] = total_reward
        
        # Mensaje extendido en consola para monitorear cómo termina cada bloque
        if (i + 1) % 500 == 0:
            resultado = "Llegó al objetivo (¡Ganó!)" if terminated else "Límite de movimientos (Truncado)"
            print(f'Episodio {i + 1} - Resultado: {resultado} - Recompensa total: {rewards_per_episode[i]:.1f} - Epsilon: {epsilon:.2f}')
        
    env.close()
    
    # Gráfica de rendimiento
    moving_avg_rewards = np.zeros(episodes)
    for t in range(episodes):
        moving_avg_rewards[t] = np.mean(rewards_per_episode[max(0, t - 100):(t + 1)])
    
    plt.title("Progreso del Entrenamiento en Acrobot-v1")
    plt.xlabel("Episodios")
    plt.ylabel("Recompensa Promedio (Últimos 100)")
    plt.plot(moving_avg_rewards)
    plt.show()
    
if __name__ == '__main__':
    train(5000)

Episodio 500 - Resultado: Límite de movimientos (Truncado) - Recompensa total: -500.0 - Epsilon: 0.80


KeyboardInterrupt: 